# OR Intelligence: train the instrument detector (YOLO) on Kaggle (or Colab)
Before running: **Settings > Accelerator: GPU (T4 x2 or P100)** and **Settings > Internet: On** (needed for pip and the download).
Data: the team's shared folder (CholecInstanceSeg boxes, 6 tools, split by surgery). CC BY-NC-SA 4.0, course use only: do not share or commit it.
When training ends, `best.pt` is in the **Output** tab (right side) under `/kaggle/working/`.

In [ ]:
!pip -q install ultralytics gdown
import os
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
print('working in', BASE)

In [ ]:
import gdown
FOLDER = 'https://drive.google.com/drive/folders/1URRLwwd3GqqY6BrDmgTMGIa4UskOIdrL'   # the shared "data" folder
gdown.download_folder(FOLDER, output=f'{BASE}/dl', quiet=False, remaining_ok=True)
!ls -la {BASE}/dl

In [ ]:
!mkdir -p {BASE}/data && cd {BASE}/data && unzip -q -o {BASE}/dl/yolo_dataset.zip
!rm -f {BASE}/dl/yolo_dataset.zip
!ls {BASE}/data && cat {BASE}/data/dataset.yaml

In [ ]:
# point the yaml at this machine's folder, keep everything else as shipped
import yaml, pathlib
p = pathlib.Path(f'{BASE}/data/dataset.yaml'); d = yaml.safe_load(p.read_text())
d['path'] = f'{BASE}/data'; p.write_text(yaml.safe_dump(d)); print(d)

## Train
`yolo11s` at 640 px, 40 epochs. On a Kaggle T4/P100 expect roughly 1-2 hours; use `epochs=25` or `yolo11n.pt` if time is short.
Classes are very unbalanced (scissors 54x rarer than grasper), so judge it by **AP per class**.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(data=f'{BASE}/data/dataset.yaml', epochs=40, imgsz=640, batch=16, patience=10,
            project=f'{BASE}/runs', name='or_tools', workers=2, device=0)

## Evaluate on the TEST split (never tuned on it)

In [ ]:
best = YOLO(f'{BASE}/runs/or_tools/weights/best.pt')
m = best.val(data=f'{BASE}/data/dataset.yaml', split='test')
print('mAP50', round(m.box.map50, 3), 'mAP50-95', round(m.box.map, 3))
for i, ap in zip(m.box.ap_class_index, m.box.ap50):
    print(f'{best.names[int(i)]:<10} AP50 {ap:.3f}')

In [ ]:
# keep only what you need in the Output tab: best.pt + the training curves, drop the 4 GB dataset
import shutil
shutil.copy(f'{BASE}/runs/or_tools/weights/best.pt', f'{BASE}/best.pt')
shutil.copy(f'{BASE}/runs/or_tools/results.csv', f'{BASE}/results.csv')
shutil.rmtree(f'{BASE}/data', ignore_errors=True); shutil.rmtree(f'{BASE}/dl', ignore_errors=True)
print(os.listdir(BASE))